<a href="https://colab.research.google.com/github/Abhi-pan1982/ML_NLP/blob/main/Marian_MT_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Machine Translation

Import Necessary Libraries

In [ ]:
#Preprocessing Libraries
import torch
import pandas as pd
import nltk
import spacy

#Evaluation Libraries
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

#Transformer Libraries for MarianMT

from transformers import MarianMTModel, MarianTokenizer



Initialize the Marian MT Model

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-ROMANCE"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/779k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/799k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.46M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Translate using Pre trained Model

In [ ]:
def translate(text):
  inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
  outputs = model.generate(**inputs)
  translated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)
  return translated_text[0]

In [ ]:
text = "How are you"
translated_text = translate(text)
print(translated_text)

- Comment allez-vous?


In [ ]:
from google.colab import files
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)
df.head()


Saving opus_books_en-fr_test.csv to opus_books_en-fr_test (7).csv


,en,fr
0,She looked out the window.,Elle regarda par la fenêtre.
1,He opened the book and started reading.,Il ouvrit le livre et commença à lire.
2,The sun was shining brightly.,Le soleil brillait de mille feux.
3,They walked along the river.,Ils marchaient le long de la rivière.
4,It was a quiet afternoon.,C'était un après-midi tranquille.


In [ ]:
preds = [translate(en) for en in df['fr']]
preds


['Elle regarde par la fenêtre.',
 'Il ouvre le livre et commença à lire.',
 'Le soleil brillant de mille feux.',
 'Ils marchent le long de la rivière.',
 "C'est un après-midi tranquille.",
 'Elle porte une écharpe rouge.',
 'Il lui chuchota un secret.',
 'Les enfants riait et jouait.',
 'La pièce sentit les fleurs fraîches.',
 'Elle tourna la page lentement.']

In [ ]:
actual = list(df['fr'])
actual

['Elle regarda par la fenêtre.',
 'Il ouvrit le livre et commença à lire.',
 'Le soleil brillait de mille feux.',
 'Ils marchaient le long de la rivière.',
 "C'était un après-midi tranquille.",
 'Elle portait une écharpe rouge.',
 'Il lui chuchota un secret.',
 'Les enfants riaient et jouaient.',
 'La pièce sentait les fleurs fraîches.',
 'Elle tourna la page lentement.']

Evaluation Matrix Using symentic simmilarity

In [ ]:
semantic_model = SentenceTransformer('sentence-transformers/LaBSE')

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

In [ ]:
## To generate tokenized input for sentence transformer for French Text

!python -m spacy download fr_core_news_sm
nlp_fr = spacy.load('fr_core_news_sm')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 28.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
def normalize_fr(text):
  doc = nlp_fr(text)
  return " ".join([token.lemma_ for token in doc])

In [ ]:
normalize_fr('Elle regarda par la fenêtre.')

'lui regarder par le fenêtre .'

In [ ]:
def check_semantic(ref, pred):
  #Normalize actual text
  ref = normalize_fr(ref)
  #Normalize our predicted text
  pred = normalize_fr(pred)

  #Generate embedding for both Normalize actual text and predicted Text

  embs = semantic_model.encode([ref, pred])

  # Check the cosine similarity between both of the embedding

  return util.cos_sim(embs[0], embs[1]).item()

In [ ]:
actual[0]

'Elle regarda par la fenêtre.'

In [ ]:
preds[0]

'Ella miró por la ventana.'

In [ ]:
check_semantic(actual[0], preds[0])

1.0000001192092896

In [ ]:
for i in range(len(df['en'])):
  ref = df['fr'].iloc[i]
  pred = preds[i]

  print('\n Similarity Score:------->', check_semantic(ref, pred))


 Similarity Score:-------> 1.0000001192092896

 Similarity Score:-------> 1.0

 Similarity Score:-------> 0.9255174398422241

 Similarity Score:-------> 1.0

 Similarity Score:-------> 1.0

 Similarity Score:-------> 0.9693260788917542

 Similarity Score:-------> 1.000000238418579

 Similarity Score:-------> 0.9609556198120117

 Similarity Score:-------> 0.9449254274368286

 Similarity Score:-------> 1.0
